# Data Cleaning: Animal Observations

In [39]:
## 1. Import libraries 

from pathlib import Path

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

## 2. Project Paths

PROJECT_DIR = Path.cwd().parent
RAW_DATA_DIR = PROJECT_DIR / "data" / "raw"

list(RAW_DATA_DIR.iterdir())

## 3. Load Dataset

csv_file = list(RAW_DATA_DIR.glob("*.csv"))[0]

df = pd.read_csv(csv_file, sep=";")

df.head()

## 4. Dataset Overview
print(f"Dataset: {csv_file.name}")
print(f"Rows: {df.shape[0]:,}")
print(f"Columns: {df.shape[1]}")

Dataset: animal_data_dirty1.csv
Rows: 1,011
Columns: 11


In [40]:
# 5. Data Quality Assessment

quality_report = pd.DataFrame({
    "null_count": df.isnull().sum(),
    "null_percentage": df.isnull().mean() * 100,
    "dtype": df.dtypes.astype(str),
    "unique_values": df.nunique()
})

quality_report

,null_count,null_percentage,dtype,unique_values
Animal type,20,1.978239,str,12
Country,12,1.186944,str,14
Weight kg,27,2.670623,float64,195
Body Length cm,27,2.670623,float64,112
Gender,19,1.879327,str,3
Animal code,1011,100.000000,float64,0
Latitude,98,9.693373,float64,735
Longitude,98,9.693373,float64,743
Animal name,959,94.856578,str,10
Observation date,0,0.000000,str,114


In [41]:
# 6. Dataset Structure

df.info()

# 7. Missing Values

missing_values = pd.DataFrame({
    "null_count": df.isnull().sum(),
    "null_percentage": df.isnull().mean().mul(100).round(2)
})

missing_values


<class 'pandas.DataFrame'>
RangeIndex: 1011 entries, 0 to 1010
Data columns (total 11 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   Animal type       991 non-null    str    
 1   Country           999 non-null    str    
 2   Weight kg         984 non-null    float64
 3   Body Length cm    984 non-null    float64
 4   Gender            992 non-null    str    
 5   Animal code       0 non-null      float64
 6   Latitude          913 non-null    float64
 7   Longitude         913 non-null    float64
 8   Animal name       52 non-null     str    
 9   Observation date  1011 non-null   str    
 10  Data compiled by  1011 non-null   str    
dtypes: float64(5), str(6)
memory usage: 87.0 KB


,null_count,null_percentage
Animal type,20,1.98
Country,12,1.19
Weight kg,27,2.67
Body Length cm,27,2.67
Gender,19,1.88
Animal code,1011,100.00
Latitude,98,9.69
Longitude,98,9.69
Animal name,959,94.86
Observation date,0,0.00


In [42]:
# 8. Duplicate Rows

duplicate_count = df.duplicated().sum()

print(f"Duplicate rows: {duplicate_count:,}")

# 9. Unique Values

unique_values = df.nunique()

unique_values

# 10. Numerical Summary

numerical_summary = df.describe().T

# 11. Data Types

data_types = df.dtypes

print(f"Unique Values per column: {unique_values}")
print(f"Numerical Summary: {numerical_summary}")
print(f"Types: {data_types}") 

Duplicate rows: 167
Unique Values per column: Animal type          12
Country              14
Weight kg           195
Body Length cm      112
Gender                3
Animal code           0
Latitude            735
Longitude           743
Animal name          10
Observation date    114
Data compiled by      4
dtype: int64
Numerical Summary:                 count       mean         std        min        25%        50%  \
Weight kg       984.0  39.745503  156.290076  -0.252000   0.293000   0.331500   
Body Length cm  984.0  39.107724   58.628601 -19.000000  19.000000  21.000000   
Animal code       0.0        NaN         NaN        NaN        NaN        NaN   
Latitude        913.0  49.393369    7.168900 -78.582973  48.186913  49.560723   
Longitude       913.0  18.203280    3.899601  11.074008  14.384559  18.944015   

                      75%          max  
Weight kg        0.800000  1100.000000  
Body Length cm  23.000000   350.000000  
Animal code           NaN          NaN  
Latitud

In [43]:
# 12. Missing Data Handling

categorical_columns = [
    "Animal type",
    "Country",
    "Gender"
]

numeric_columns = [
    "Weight kg",
    "Body Length cm",
    "Latitude",
    "Longitude"
]

for column in categorical_columns:
    print(f"{column} mode: {df[column].mode()[0]}")

for column in numeric_columns:
    print(f"{column} median: {df[column].median()}")

Animal type mode: red squirrel
Country mode: Poland
Gender mode: male
Weight kg median: 0.3315
Body Length cm median: 21.0
Latitude median: 49.560723
Longitude median: 18.944015


In [44]:
# 13. Apply Missing Data Treatment

# Impute categorical missing values with the mode
for column in categorical_columns:
    df[column] = df[column].fillna(df[column].mode()[0])

# Impute numerical missing values with the median
for column in numeric_columns:
    df[column] = df[column].fillna(df[column].median())

# Remove the fully missing Animal code column
df = df.drop(columns = ["Animal code"])

# 14. Verify Missing Values After Treatment

missing_after = pd.DataFrame({
    "null_count": df.isnull().sum(),
    "null_percentage": df.isnull().mean().mul(100).round(2)
})

missing_after

,null_count,null_percentage
Animal type,0,0.00
Country,0,0.00
Weight kg,0,0.00
Body Length cm,0,0.00
Gender,0,0.00
Latitude,0,0.00
Longitude,0,0.00
Animal name,959,94.86
Observation date,0,0.00
Data compiled by,0,0.00


## Missing Data Handling Rationale

- **Categorical variables:** Missing values in `Animal type`, `Country`, and `Gender` were imputed using the mode because these variables contain categorical information.
- **Numeric variables:** Missing values in `Weight kg`, `Body Length cm`, `Latitude`, and `Longitude` were imputed using the median because the median is less affected by extreme values.
- **Animal code:** The column was removed because all 1,011 values were missing and therefore contained no usable information.
- **Animal name:** Missing values were retained because 94.86% of the values were missing, making reliable imputation inappropriate.

In [46]:
# 16. Remove Duplicate Rows

rows_before = len(df)

df = df.drop_duplicates().reset_index(drop=True)

rows_after = len(df)
duplicates_removed = rows_before - rows_after

print(f"Rows before: {rows_before:,}")
print(f"Rows after: {rows_after:,}")
print(f"Duplicates removed: {duplicates_removed:,}")

Rows before: 1,011
Rows after: 844
Duplicates removed: 167


## Duplicate Removal

A total of 167 duplicate rows were identified and removed from the dataset.

The dataset was reduced from 1,011 rows to 844 rows. The index was reset after removing the duplicates.